In [1]:
from moe_reft.modeling_olmoe import OlmoeForCausalLM
from moe_reft import configuration_olmoe

model = OlmoeForCausalLM(configuration_olmoe.OlmoeInterventionsConfig(
    interventions_config=configuration_olmoe.InterventionsConfig(
        intervention_places="after_moe",
        intervention_layers="even_only",
    ),
))

/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Mapping, MutableMapping, Sequence

import fnmatch
import torch
from loguru import logger
from torch import nn
from transformers import AutoModelForCausalLM, PreTrainedModel

map_dtype=torch.bfloat16  # optional casting
map_device=torch.device("cuda")  # optional device move
        
hf_model_name_or_path="allenai/OLMoE-1B-7B-0125-Instruct"
# hf_model_name_or_path="allenai/OLMoE-1B-7B-0924-Instruct"

hf_model: PreTrainedModel = AutoModelForCausalLM.from_pretrained(
        hf_model_name_or_path,
        dtype=map_dtype if map_dtype is not None else None,
        trust_remote_code=True,
    )

src_sd = hf_model.state_dict()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
from moe_reft import load_weights

intervention_patterns = [
            "*.pre_moe_intervention.*",
            "*.after_moe_intervention.*",
            "*.pre_moe_intervenetion.*",  # in case of typos in existing checkpoints
        ]

dst_sd: MutableMapping[str, torch.Tensor] = model.state_dict()
out: dict[str, torch.Tensor] = {}


In [8]:
for src_name, src_tensor in dst_sd.items():
    if "experts" in src_name:
        break

In [9]:
src_name, src_tensor.shape

('model.layers.0.mlp.experts.0.gate_proj.weight', torch.Size([2048, 2048]))

In [5]:
report = load_weights.TransferReport.from_empty()
dtype = torch.bfloat16
device = "cuda"

for src_name, src_tensor in src_sd.items():
    if load_weights._matches_any(src_name, intervention_patterns):
        report.skipped_intervention.append(src_name)
        continue
    cand_names: list[str] = []
    
    
    matched_dst: str | None = next((n for n in [src_name] if n in dst_sd), None)
    
    if matched_dst is None:
        report.skipped_missing.append(src_name)
        logger.info(f"Missing for {matched_dst=} with {src_name=}")
        continue

    if load_weights._matches_any(matched_dst, intervention_patterns):
        report.skipped_intervention.append(src_name)
        continue
    
    dst_t: torch.Tensor = dst_sd[matched_dst]
    
    if src_tensor.shape != dst_t.shape:
        report.skipped_shape.append((matched_dst, src_tensor.shape, dst_t.shape))
        logger.error(
            f"The shape for {src_name=} with {src_tensor.shape=} didn't match with for {matched_dst=} and shape {dst_t.shape=}"
        )
        continue
    
    if dtype is not None or device is not None:
            src_tensor = src_tensor.to(
                device=device if device is not None else dst_t.device,
                dtype=dtype if dtype is not None else dst_t.dtype,
            )
    out[matched_dst] = src_tensor
    report.copied.append(matched_dst)
        

2025-11-06 03:19:13.768 | ERROR    | __main__:<module>:27 - The shape for src_name='model.layers.0.mlp.experts.0.gate_proj.weight' with src_tensor.shape=torch.Size([1024, 2048]) didn't match with for matched_dst='model.layers.0.mlp.experts.0.gate_proj.weight' and shape dst_t.shape=torch.Size([2048, 2048])
2025-11-06 03:19:13.769 | ERROR    | __main__:<module>:27 - The shape for src_name='model.layers.0.mlp.experts.0.up_proj.weight' with src_tensor.shape=torch.Size([1024, 2048]) didn't match with for matched_dst='model.layers.0.mlp.experts.0.up_proj.weight' and shape dst_t.shape=torch.Size([2048, 2048])
2025-11-06 03:19:13.769 | ERROR    | __main__:<module>:27 - The shape for src_name='model.layers.0.mlp.experts.0.down_proj.weight' with src_tensor.shape=torch.Size([2048, 1024]) didn't match with for matched_dst='model.layers.0.mlp.experts.0.down_proj.weight' and shape dst_t.shape=torch.Size([2048, 2048])
2025-11-06 03:19:13.770 | ERROR    | __main__:<module>:27 - The shape for src_name=